In [21]:
import pandas as pd

# 1. Dosya yolunu data klasörünü gösterecek şekilde güncelliyoruz
df = pd.read_csv('../data/raw/train.csv') 

# 2. Temizlik işlemleri için bir kopyasını oluşturuyoruz
df_clean = df.copy()

# Başarıyla kopyalandığını teyit edelim
df_clean.head(3)

,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied


In [22]:
df_clean.drop(columns=["Unnamed: 0", "id"], inplace=True) # veri setindeki gereksiz sütunları kaldırır.

In [14]:
# 1. Tüm harfleri küçült
# 2. Boşlukları alt çizgi yap
# 3. Eğik çizgi (/) işaretlerini alt çizgi yap
# 4. Tire (-) işaretlerini alt çizgi yap

df_clean.columns = (
    df_clean.columns
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
    .str.replace('-', '_')
)

# sütun isimlerini yazdıralım:

print(df_clean.columns)


Index(['gender', 'customer_type', 'age', 'type_of_travel', 'class',
       'flight_distance', 'inflight_wifi_service',
       'departure_arrival_time_convenient', 'ease_of_online_booking',
       'gate_location', 'food_and_drink', 'online_boarding', 'seat_comfort',
       'inflight_entertainment', 'on_board_service', 'leg_room_service',
       'baggage_handling', 'checkin_service', 'inflight_service',
       'cleanliness', 'departure_delay_in_minutes', 'arrival_delay_in_minutes',
       'satisfaction'],
      dtype='object')


### Sütun İsimlerinin Standardizasyonu

Veri seti üzerinde yapılan incelemede, sütun isimlerinin büyük/küçük harf tutarsızlıkları, boşluklar ve özel karakterler ('/', '-') içerdiği görülmüştür (Örn: `Customer Type`, `Departure/Arrival time convenient`). 

Veri manipülasyonu ve makine öğrenmesi modelleme aşamalarında kod okunabilirliğini artırmak, sözdizimi (syntax) hatalarını önlemek ve metodolojik standartları sağlamak amacıyla tüm sütun isimleri **"snake_case"** formatına dönüştürülmüştür.

**Uygulanan Dönüşümler:**
* Tüm harfler küçük harf formatına (`lower()`) dönüştürüldü.
* İsimler arasındaki boşluklar, tire (`-`) ve eğik çizgi (`/`) işaretleri, alt çizgi (`_`) ile değiştirildi.


# Veri Tiplerinin Optimizasyonu

In [15]:
# Kategorik (metin içeren) sütunları seçiyoruz
kategorik_sutunlar = ['gender', 'customer_type', 'type_of_travel', 'class', 'satisfaction']

# Bir döngü ile hepsini tek tek 'category' tipine dönüştürüyoruz
for sutun in kategorik_sutunlar:
    df_clean[sutun] = df_clean[sutun].astype('category')

df_clean.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103904 entries, 0 to 103903
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype   
---  ------                             --------------   -----   
 0   gender                             103904 non-null  category
 1   customer_type                      103904 non-null  category
 2   age                                103904 non-null  int64   
 3   type_of_travel                     103904 non-null  category
 4   class                              103904 non-null  category
 5   flight_distance                    103904 non-null  int64   
 6   inflight_wifi_service              103904 non-null  int64   
 7   departure_arrival_time_convenient  103904 non-null  int64   
 8   ease_of_online_booking             103904 non-null  int64   
 9   gate_location                      103904 non-null  int64   
 10  food_and_drink                     103904 non-null  int64   
 11  online_boarding           

###  Veri Tipi Optimizasyonu (Memory Optimization)

Veri manipülasyonu aşamasında bellek (RAM) kullanımını minimize etmek ve ilerleyen fazlarda kurulacak makine öğrenmesi algoritmalarının eğitim hızını artırmak hedeflenmiştir. 

Bu doğrultuda, az sayıda farklı eşsiz değere sahip metinsel özellikler (`object` tipindeki sütunlar), istatistiksel modelleme prensiplerine uygun olarak `category` veri tipine dönüştürülmüştür. Bu işlem sayesinde veri setinin sistem üzerindeki yükü ciddi oranda hafifletilmiştir.

### Tekrarlayan Veri Kontrolü

In [16]:
# Önce veri setinde birbirinin tamamen aynısı kaç satır var onu bulalım
kopya_sayisi = df_clean.duplicated().sum()
print(f"Tespit edilen kopya satır sayısı: {kopya_sayisi}")

# Eğer kopya satır varsa, bunları veri setinden kalıcı olarak siliyoruz
df_clean.drop_duplicates(inplace=True)

# Temizlik sonrası elimizde kalan net satır sayısını kontrol edelim
print(f"Güncel ve temizlenmiş satır sayımız: {len(df_clean)}")

Tespit edilen kopya satır sayısı: 0
Güncel ve temizlenmiş satır sayımız: 103904


### Tekrarlayan Verilerin Kontrolü

Makine öğrenmesi modellerinde aşırı öğrenmeye (overfitting) ve algoritmik yanlılığa (bias) sebep olabilecek mükerrer kayıtlar `df.duplicated()` metodu ile incelenmiştir.

**Sonuç ve Çıkarım:**
Yapılan tarama sonucunda veri setinde **0 adet kopya satır** tespit edilmiştir. Mevcut 103.904 satırın tamamının eşsiz (unique) olduğu doğrulanmıştır. Veri toplama sürecinin oldukça temiz yürütüldüğünü ve veri bütünlüğünün kusursuz olduğunu gösteren bu sonuç sayesinde, herhangi bir satır silme işlemine gerek kalmadan bir sonraki analitik aşamaya geçilmiştir.